# Pipeline Monitor
Quick checkpoints for the LEAGUE_RECORDS pipeline. Run All to get a snapshot.

In [ ]:
%%sql -r ctx
USE WAREHOUSE COMPUTE_WH;
USE DATABASE LEAGUE_RECORDS;

In [ ]:
%%sql -r load_state
-- Where are we in the simulation?
SELECT
    CURRENT_LOAD_DATE,
    MIN_DATE,
    MAX_DATE,
    LAST_LOADED_AT,
    DATEDIFF('day', CURRENT_LOAD_DATE, MIN_DATE) AS DAYS_REMAINING
FROM SEED.SEED_LOAD_STATE;

In [ ]:
-- Row counts across all layers
SELECT 
    'SEED' AS LAYER,
    'MATCHES' AS TABLE_NAME, 
    COUNT(*) AS ROWS_COUNT
FROM SEED.SEED_MATCHES_SUMMARY
UNION ALL SELECT 'BRONZE', 'MATCHES', COUNT(*) FROM BRONZE.MATCHES_SUMMARY_BRONZE
UNION ALL SELECT 'SILVER', 'MATCHES', COUNT(*) FROM SILVER.MATCHES_SUMMARY_SILVER
UNION ALL SELECT 'BRONZE', 'PLAYERS', COUNT(*) FROM BRONZE.PLAYERS_SUMMARY_BRONZE
UNION ALL SELECT 'SILVER', 'PLAYERS', COUNT(*) FROM SILVER.PLAYERS_SUMMARY_SILVER
UNION ALL SELECT 'BRONZE', 'INTERVALS', COUNT(*) FROM BRONZE.MATCH_INTERVALS_BRONZE
UNION ALL SELECT 'SILVER', 'TEAM_INTERVALS', COUNT(*) FROM SILVER.TEAM_INTERVAL_SILVER
UNION ALL SELECT 'SILVER', 'PLAYER_INTERVALS', COUNT(*) FROM SILVER.PLAYER_INTERVAL_SILVER;

In [ ]:
%%sql -r task_history
-- Task execution history (last 2 hours)
SELECT
    NAME,
    STATE,
    SCHEDULED_TIME,
    COMPLETED_TIME,
    DATEDIFF('second', SCHEDULED_TIME, COMPLETED_TIME) AS DURATION_SEC,
    ERROR_CODE,
    ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD('hour', -2, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 20
))
ORDER BY SCHEDULED_TIME DESC;

In [ ]:
%%sql -r pipe_usage
-- Pipe usage (last 2 hours)
SELECT
    START_TIME,
    PIPE_NAME,
    CREDITS_USED,
    BYTES_INSERTED,
    FILES_INSERTED
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.PIPE_USAGE_HISTORY(
    DATE_RANGE_START => DATEADD('hour', -2, CURRENT_TIMESTAMP()),
    DATE_RANGE_END => CURRENT_TIMESTAMP()
))
WHERE PIPE_NAME IS NOT NULL
ORDER BY START_TIME DESC;

In [ ]:
%%sql -r stream_status
-- Stream status: any unconsumed data?
SELECT
    'MATCHES' AS STREAM,
    SYSTEM$STREAM_HAS_DATA('BRONZE.MATCHES_SUMMARY_BRONZE_STM') AS HAS_DATA
UNION ALL SELECT 'PLAYERS', SYSTEM$STREAM_HAS_DATA('BRONZE.PLAYERS_SUMMARY_BRONZE_STM')
UNION ALL SELECT 'INTERVALS', SYSTEM$STREAM_HAS_DATA('BRONZE.MATCH_INTERVALS_BRONZE_STM')
UNION ALL SELECT 'ITEMS_REF', SYSTEM$STREAM_HAS_DATA('BRONZE.ITEMS_REF_BRONZE_STM')
UNION ALL SELECT 'CHAMPIONS_REF', SYSTEM$STREAM_HAS_DATA('BRONZE.CHAMPIONS_REF_BRONZE_STM');

In [ ]:
SELECT 
    'PLAYER_INTERVALS' AS TABLE_NAME,
    COUNT(*) AS ROW_COUNT
FROM SILVER.PLAYER_INTERVAL_SILVER
    UNION ALL
SELECT
    'TEAM_INTERVALS' AS TABLE_NAME,
    COUNT(*) AS ROW_COUNT
FROM SILVER.TEAM_INTERVAL_SILVER
;

In [ ]:
SELECT * 
FROM SILVER.PLAYER_INTERVAL_SILVER
ORDER BY RANDOM()
LIMIT 5
;